Análise de Dados Climáticos de 2024
1. Importar as bibliotecas

As bibliotecas usadas foram pandas e altair.

In [14]:
import pandas as pd
import altair as alt

alt.data_transformers.disable_max_rows()

DataTransformerRegistry.enable('default')

2. Carregar e tratar os dados

Os dados foram obtidos a partir de arquivos CSV disponibilizados pelo Instituto Nacional de Meteorologia (INMET), sendo referentes ao ano de 2024 para possibilitar uma análise detalhada de todos os fatores que influenciaram os eventos meteorológicos extremos daquele ano. Fora isso, foram renomeados os atributos, para facilitar sua visualização, além da data e hora que foram convertidas em datetime.

In [15]:
file_1 = "dados_climaticos_belem_novo.csv"
file_2 = "dados_climaticos_jardim_botanico.csv"

def carregar_dados(file, bairro_nome):
    # Ler CSV
    df = pd.read_csv(file, sep=';', decimal=',', encoding='latin1', skiprows=8, header=0)

    # Visualizar primeiras linhas
    print(df.head())

    # Renomear colunas para facilitar uso
    df.columns = [
        "data", "hora", "precipitacao", "pressao",
        "pressao_max", "pressao_min", "radiacao",
        "temperatura", "ponto_orvalho", "temp_max",
        "temp_min", "orvalho_max", "orvalho_min",
        "umidade_max", "umidade_min", "umidade",
        "vento_direcao", "vento_rajada", "vento_velocidade", "extra"
    ]

    # Converter data + hora em datetime
    df['hora'] = df['hora'].str.replace(' UTC', '', regex=False)
    df['datetime'] = pd.to_datetime(df['data'] + ' ' + df['hora'], format='%Y/%m/%d %H%M')

    # Remover colunas antigas se quiser
    df = df.drop(columns=['data', 'hora'])
    df['bairro'] = bairro_nome

    # Remover medições completamente nulas
    return df.dropna(how='all')

df1 = carregar_dados(file_1, "Belem Novo")
df2 = carregar_dados(file_2, "Jardim Botanico")

df = pd.concat([df1, df2], ignore_index=True)

print(df.shape)

         Data  Hora UTC  PRECIPITAÇÃO TOTAL, HORÁRIO (mm)  \
0  2024/01/01  0000 UTC                               0.0   
1  2024/01/01  0100 UTC                               0.0   
2  2024/01/01  0200 UTC                               0.0   
3  2024/01/01  0300 UTC                               0.0   
4  2024/01/01  0400 UTC                               0.0   

   PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)  \
0                                             1015.9       
1                                             1016.5       
2                                             1016.4       
3                                             1015.8       
4                                             1015.4       

   PRESSÃO ATMOSFERICA MAX.NA HORA ANT. (AUT) (mB)  \
0                                           1015.9   
1                                           1016.5   
2                                           1016.6   
3                                           1016.4   
4 

3. Agregação dos dados

Para este conjunto de dados, com o intuito de auxiliar na compreensão dos gráficos, foi utilizada uma medida de agregação ao empregar a média entre todos os valores obtidos em determinado dia. 

In [35]:
# Conferir estrutura
print(df.info())

# Análise exploratória simples

print(df.describe())

# Convertendo para médias diárias (para suavizar e facilitar visualização)

df_daily = df.resample('D', on='datetime').mean(numeric_only=True).reset_index()

# Para correlação com bairro, criar df_daily_bairro
df_daily_bairro = df.groupby(['bairro', pd.Grouper(key='datetime', freq='D')]).mean(numeric_only=True).reset_index()

# Criar coluna de mês (para filtro)
df_daily['mes'] = df_daily['datetime'].dt.month

<class 'pandas.DataFrame'>
RangeIndex: 17568 entries, 0 to 17567
Data columns (total 20 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   precipitacao      17550 non-null  float64       
 1   pressao           17549 non-null  float64       
 2   pressao_max       17538 non-null  float64       
 3   pressao_min       17538 non-null  float64       
 4   radiacao          13503 non-null  float64       
 5   temperatura       17549 non-null  float64       
 6   ponto_orvalho     17548 non-null  float64       
 7   temp_max          17538 non-null  float64       
 8   temp_min          17538 non-null  float64       
 9   orvalho_max       17538 non-null  float64       
 10  orvalho_min       17537 non-null  float64       
 11  umidade_max       17538 non-null  float64       
 12  umidade_min       17537 non-null  float64       
 13  umidade           17548 non-null  float64       
 14  vento_direcao     17546 non-null 

4. Visualizações com Altair

Possibilitamos ao usuário interagir de diversas maneiras distintas com cada gráfico, tornando o ato de comparar as diferentes medidas obtidas em cada um mais dinâmico e divertido.

In [36]:
# Criar seleção de mês

mes_selecao = alt.selection_point(fields=['mes'], bind=alt.binding_range(min=1, max=12, step=1, name="Mês"), value=1)
intervalo_selecao = alt.selection_interval(encodings=['x'], name="Intervalo")

# Visualizações com Altair

# Temperatura ao longo do tempo

media_temp = df_daily['temperatura'].mean()

bar_temp = alt.Chart(df_daily).mark_bar().encode(
    x='datetime:T',
    y='temperatura:Q'
).add_params(mes_selecao).transform_filter(mes_selecao)

linha_media = alt.Chart(pd.DataFrame({'y': [media_temp]})).mark_rule(color='red').encode(y='y:Q')

chart_temp = (bar_temp + linha_media).properties(title='Temperatura média diária com média global')

chart_temp

alt.LayerChart(...)

Precipitação

In [37]:
base = alt.Chart(df_daily).mark_bar(color='purple').encode(
    x='datetime:T',
    y='precipitacao:Q'
)

chart_precip = base.add_params(intervalo_selecao).properties(title='Seleção de intervalo - precipitação')

zoomed = base.transform_filter(intervalo_selecao)

chart_precip_final = chart_precip | zoomed
chart_precip_final

alt.HConcatChart(...)

Umidade

In [38]:
chart_umidade = alt.Chart(df_daily).mark_line(color='green').encode(
    x='datetime:T',
    y='umidade:Q',
    tooltip=['datetime', 'umidade']
).properties(
    title='Umidade relativa do ar'
).interactive()

chart_umidade

alt.Chart(...)

🌡️ Comparação temperatura min/max

In [39]:
chart_temp_range = alt.Chart(df_daily).transform_fold(
    ['temp_min', 'temp_max'],
    as_=['tipo', 'valor']
).mark_line().encode(
    x='datetime:T',
    y='valor:Q',
    color='tipo:N'
).properties(
    title='Temperatura mínima vs máxima'
).interactive()

chart_temp_range

# =========================================
# 6. Possíveis melhorias
# =========================================

# - Filtrar por período - sim
# - Criar médias diárias - com certeza
# - Criar dashboard com múltiplos gráficos

alt.Chart(...)

Temperatura média diária

In [40]:
chart_temp = alt.Chart(df_daily).mark_line().encode(
    x='datetime:T',
    y='temperatura:Q',
    tooltip=['datetime', 'temperatura']
).add_params(
    mes_selecao
).transform_filter(
    mes_selecao
).properties(
    title='Temperatura média diária'
).interactive()

chart_temp

alt.Chart(...)

🔥 Dias mais quentes e frios

In [41]:
chart_extremos = alt.Chart(df_daily).mark_point(size=60).encode(
    x='datetime:T',
    y='temperatura:Q',
    color=alt.condition(
        alt.datum.temperatura > df_daily['temperatura'].mean(),
        alt.value('red'),
        alt.value('blue')
    ),
    tooltip=['datetime', 'temperatura']
).add_params(
    mes_selecao
).transform_filter(
    mes_selecao
).properties(
    title='Dias mais quentes (vermelho) vs frios (azul)'
)

chart_extremos

alt.Chart(...)

Histogramas

In [42]:
hist_temp = alt.Chart(df_daily).mark_bar(color='lightblue').encode(
    alt.X('temperatura:Q', bin=True),
    y='count()'
).properties(title='Histograma de Temperatura')

hist_umidade = alt.Chart(df_daily).mark_bar(color='lightgreen').encode(
    alt.X('umidade:Q', bin=True),
    y='count()'
).properties(title='Histograma de Umidade')

hist_temp | hist_umidade

alt.HConcatChart(...)

💧 Correlação temperatura vs umidade

In [43]:
chart_corr_temp_umidade = alt.Chart(df_daily_bairro).mark_circle(size=60).encode(
    x='temperatura:Q',
    y='umidade:Q',
    color='bairro:N',
    tooltip=['temperatura', 'umidade', 'bairro']
).properties(
    title='Correlação: Temperatura vs Umidade'
)
chart_corr_temp_umidade

alt.Chart(...)

💨 Correlação vento (rajada) vs precipitação

In [44]:
chart_corr_vento = alt.Chart(df_daily_bairro).mark_circle(size=60).encode(
    x='vento_rajada:Q',
    y='precipitacao:Q',
    color='bairro:N',
    tooltip=['vento_rajada', 'precipitacao', 'bairro']
).properties(
    title='Correlação: Rajada de vento vs Precipitação'
)
chart_corr_vento

alt.Chart(...)

Precipitação vs Temperatura

In [47]:
scatter_precip_temp = alt.Chart(df_daily_bairro).mark_circle(size=60).encode(
    x='precipitacao:Q',
    y='temperatura:Q',
    color='bairro:N',
    tooltip=['vento_rajada', 'precipitacao', 'bairro']
).properties(title='Precipitação vs Temperatura')
scatter_precip_temp

alt.Chart(...)

Temperatura média entre bairros

In [ ]:
bar_bairros = alt.Chart(df).mark_bar().encode(
    y='bairro:N',
    x='mean(temperatura):Q',
    color='bairro:N'
).properties(title='Temperatura média por bairro')
bar_bairros

alt.Chart(...)

Correlação entre múltiplas variáveis (Temperetaura, umidade, precipitação, velocidade do vento)

In [52]:
multi_scatter = alt.Chart(df_daily).mark_circle().encode(
    x='temperatura:Q',
    y='umidade:Q',
    size='precipitacao:Q',
    color='vento_velocidade:Q',
    tooltip=['temperatura', 'umidade', 'precipitacao', 'vento_velocidade']
).properties(title='Multivariável')
multi_scatter

alt.Chart(...)

Agregação interativa

In [54]:
agg_chart = alt.Chart(df_daily_bairro).mark_line().encode(
    x='month(datetime):O',
    y='mean(temperatura):Q',
    color='bairro:N'
).properties(title='Média mensal interativa')
agg_chart

alt.Chart(...)

Temperatura vs Umidade (ainda mais interativo)

Como a relação entre umidade e temperatura muda acima de X °C?

In [55]:
# Slider de temperatura
slider = alt.binding_range(min=float(df_daily['temperatura'].min()),
                           max=float(df_daily['temperatura'].max()),
                           step=0.5)

threshold = alt.param(name="threshold", value=float(df_daily['temperatura'].mean()), bind=slider)

# Camada 1: pontos acima do threshold
scatter_high = alt.Chart(df_daily).mark_circle(size=60, color='blue').encode(
    x=alt.X("temperatura:Q", title="Temperatura (°C)"),
    y=alt.Y("umidade:Q", title="Umidade (%)"),
    tooltip=['temperatura', 'umidade']
).transform_filter(
    alt.datum.temperatura >= threshold
)

# Camada 2: pontos abaixo (agregados)
scatter_low = alt.Chart(df_daily).mark_circle().encode(
    x=alt.X("temperatura:Q", bin=alt.Bin(maxbins=20)),
    y=alt.Y("umidade:Q", bin=alt.Bin(maxbins=20)),
    size=alt.Size("count():Q", scale=alt.Scale(domain=[0, 100]))
).transform_filter(
    alt.datum.temperatura < threshold
)

# Linha vertical do threshold
rule = alt.Chart().mark_rule(color="black").encode(
    x=alt.X(datum=threshold, type="quantitative")
)

# Combinar tudo
chart_threshold = alt.layer(
    scatter_high,
    scatter_low,
    rule
).add_params(
    threshold
).properties(
    title="Temperatura vs Umidade com Threshold Interativo"
)

chart_threshold

alt.LayerChart(...)